# Pipeline Steps 7, 8 & 9 — Evaluation (H1, H2, H3)
**GWU Praxis, Chapter 3 (Section 3.10)**

Evaluates every saved model (Random Forest, CNN-LSTM A, CNN-LSTM B; three seeds each). No model is trained here.

**Zero-day detection definition (fixed before any result is computed).** A held-out record is *detected* when the model flags it as **any attack class**: its attack score, 1 − P(benign), exceeds the model's alert threshold. The threshold is set on the **validation** benign records so that every model flags the same share of benign traffic (`FPR_TARGET`, 1%). This is the only definition that applies to all three models, because Random Forest and CNN-LSTM A have no Synthetic-OOD output. Two other quantities are reported but are not used for any hypothesis:

- **Synthetic-OOD rate** (Experiment B only, descriptive): share of held-out records whose top prediction is Synthetic-OOD, with the false-OOD rate on known-class test records.
- **Exact held-out label:** zero for every model by construction, since no model has an output for the held-out class. Reported once to show why it is not used.

**Tests.** H1 — paired bootstrap on the gap G = F1_known − F1_zero-day (Section 3.10.1). H2 — detection recall at the common false-alarm rate; exact McNemar test, CNN-LSTM B vs A, Bonferroni α = 0.05/3. H3 — SHAP ranking similarity between real and reserved synthetic records, reported next to a real-vs-real ceiling and a real-vs-benign floor. Paired tests use, for each model, the seed with the highest validation macro-F1; all seeds are summarised as mean ± SD.

**Requires:** Steps 1–6. **Produces:** a new folder `outputs_praxis_heldout/evaluation/<run id>/` with every table, JSON file, plot and an HTML report.

## 0 · Imports and settings

In [1]:
# ────────────────────────────────────────────────────────────────────────────
# IMPORTS AND SESSION SETUP
# Each evaluation writes to a new time-stamped folder; earlier results are
# never overwritten.
# ────────────────────────────────────────────────────────────────────────────
import importlib.util, subprocess, sys
if importlib.util.find_spec("shap") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "shap"])
import os, json, pickle, re, gc, hashlib, html as html_lib, base64
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from scipy.stats import binomtest, spearmanr
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score
from IPython.display import display, HTML
import warnings
warnings.filterwarnings("ignore", message="The structure of `inputs`")
warnings.filterwarnings("ignore", category=UserWarning, module="tqdm")

# Set this to your project folder in Google Drive (it must end with "/").
BASE = os.environ.get("PRAXIS_BASE", "/content/drive/MyDrive/<YOUR_PROJECT_FOLDER>/")
RUN_DIR = "outputs_praxis_heldout"   # results folder
if "PRAXIS_BASE" not in os.environ:
    from google.colab import drive
    drive.mount("/content/drive")
ROOT = Path(BASE) / RUN_DIR
PIPE, SYN, MOD = ROOT / "pipeline", ROOT / "SynthGen", ROOT / "classifier" / "models"
EVAL_ID = "evaluation_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
OUT = ROOT / "evaluation" / EVAL_ID
OUT.mkdir(parents=True, exist_ok=False)
DATASETS = ["CIC-IDS2017", "UNSW-NB15", "NSL-KDD"]
print("Results folder:", OUT)

Mounted at /content/drive
Results folder: /content/drive/MyDrive/ColabNotebooks/GWUPraxis/outputs_praxis_heldout/evaluation/evaluation_20261024_182307


In [2]:
# ────────────────────────────────────────────────────────────────────────────
# EVALUATION SETTINGS
#
# DETECTION_DEFINITION is recorded in every output file. FPR_TARGET is the
# benign false-alarm rate all models are held to (set on validation data).
# KNOWN_F1_MARGIN: H1's largest tolerated drop in known-class macro-F1.
# ────────────────────────────────────────────────────────────────────────────
DETECTION_DEFINITION = "any_attack: 1 − P(benign) above a validation-calibrated threshold"
FPR_TARGET       = 0.01
FPR_SENSITIVITY  = [0.05]
KNOWN_F1_MARGIN  = 0.02
N_BOOT           = 2000
ALPHA            = 0.05 / len(DATASETS)          # Bonferroni, shared by H1 and H2
H3_THRESHOLD     = 0.8
RUN_SHAP         = True
SHAP_MODELS      = ["cnn_B", "cnn_A", "rf"]      # remove "rf" to save time on large forests
SHAP_N           = 30                            # records per set, per class
SHAP_BACKGROUND  = 50
SHAP_PERMUTATIONS = 1000
MODEL_NAMES = {"rf": "Random Forest (A)", "cnn_A": "CNN-LSTM A (standard)", "cnn_B": "CNN-LSTM B (augmented)"}

## 1 · Load data and every saved model

In [3]:
# ────────────────────────────────────────────────────────────────────────────
# HELPERS: JSON output and the CNN-LSTM architecture
# The architecture is rebuilt from the grid-search choice saved in Step 5.
# ────────────────────────────────────────────────────────────────────────────
def clean_json(v):
    if isinstance(v, dict): return {str(k): clean_json(x) for k, x in v.items()}
    if isinstance(v, (list, tuple)): return [clean_json(x) for x in v]
    if isinstance(v, np.ndarray): return clean_json(v.tolist())
    if isinstance(v, np.generic): return clean_json(v.item())
    if isinstance(v, float) and not np.isfinite(v): return None
    return v

def save_json(path, obj):
    Path(path).write_text(json.dumps(clean_json(obj), indent=2, allow_nan=False))

def build_cnn_lstm(input_dim, n_classes, filters, lstm_units, dropout):
    inp = keras.Input(shape=(input_dim, 1), name="input")
    x = layers.Conv1D(filters, 3, activation="relu", padding="same", name="conv1")(inp)
    x = layers.MaxPooling1D(pool_size=2, name="pool1")(x)
    x = layers.LSTM(lstm_units, name="lstm1")(x)
    x = layers.Dense(64, activation="relu", name="dense1")(x)
    x = layers.Dropout(dropout, name="dropout1")(x)
    return keras.Model(inp, layers.Dense(n_classes, activation="softmax", name="output")(x))

def file_hash(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(8 << 20), b""):
            h.update(block)
    return h.hexdigest()

In [4]:
# ────────────────────────────────────────────────────────────────────────────
# LOAD SPLITS, SYNTHETIC RESERVE, METADATA AND MODELS
#
# Checks before anything is scored:
#   • the held-out label is not a training class and is not an output of any model;
#   • Experiment B's outputs are Experiment A's plus Synthetic-OOD (last);
#   • every synthetic record is labelled Synthetic-OOD.
# Also records a hash of every input file and counts feature-identical records
# shared between partitions (should be ~0 after Step 2's de-duplication).
# ────────────────────────────────────────────────────────────────────────────
data, models, manifest, overlaps = {}, {}, {}, {}
for name in DATASETS:
    meta = pickle.load(open(MOD / f"{name}_classifier_meta.pkl", "rb"))
    fc, lab = meta["feat_cols"], meta["label_col"]
    d = dict(meta)
    for k in ["train", "val", "test_standard", "test_zeroday"]:
        p = PIPE / f"{name}_{k}.csv"
        d[k] = pd.read_csv(p); manifest[str(p)] = file_hash(p)
    p = SYN / f"{name}_synthetic_ood.csv"
    syn = pd.read_csv(p); manifest[str(p)] = file_hash(p)
    d["h3_synth"] = syn[syn["split"] == "h3_reserve"].reset_index(drop=True)

    heldout = set(d["test_zeroday"][lab].astype(str))
    assert heldout and heldout <= set(meta["holdout"]), f"{name}: unexpected held-out labels {heldout}"
    assert not heldout & set(meta["classes_a"]) and not heldout & set(meta["classes_b"]), f"{name}: held-out class is a model output"
    assert meta["classes_b"] == meta["classes_a"] + [meta["ood_label"]], f"{name}: B outputs must be A outputs + Synthetic-OOD"
    assert (syn[lab] == meta["ood_label"]).all(), f"{name}: synthetic records not labelled Synthetic-OOD"
    code = {c: i for i, c in enumerate(meta["classes_b"])}
    d["code"] = code
    d["benign_id"] = [i for i, c in enumerate(meta["classes_a"]) if c.strip().lower() in ("benign", "normal")]
    assert len(d["benign_id"]) == 1, f"{name}: benign class not found"; d["benign_id"] = d["benign_id"][0]
    d["X_val"] = d["val"][fc].to_numpy(np.float32);          d["y_val"] = d["val"][lab].map(code).to_numpy(int)
    d["X_std"] = d["test_standard"][fc].to_numpy(np.float32); d["y_std"] = d["test_standard"][lab].map(code).to_numpy(int)
    d["X_zd"] = d["test_zeroday"][fc].to_numpy(np.float32)

    hashes = {k: set(pd.util.hash_pandas_object(d[k][fc], index=False)) for k in ["train", "test_standard", "test_zeroday"]}
    overlaps[name] = dict(train_standard=len(hashes["train"] & hashes["test_standard"]),
                          train_heldout=len(hashes["train"] & hashes["test_zeroday"]),
                          standard_heldout=len(hashes["test_standard"] & hashes["test_zeroday"]))

    mm = {"rf": {}, "cnn_A": {}, "cnn_B": {}}
    for seed in meta["seeds"]:
        p = MOD / f"{name}_rf_seed{seed}.joblib"
        mm["rf"][seed] = joblib.load(p); manifest[str(p)] = file_hash(p)
        assert np.array_equal(mm["rf"][seed].classes_, np.arange(len(meta["classes_a"])))
        for exp in ["A", "B"]:
            p = MOD / f"{name}_cnn_exp{exp}_seed{seed}.weights.h5"
            m = build_cnn_lstm(len(fc), len(meta["classes_a"]) + (exp == "B"), **meta["arch"])
            m.load_weights(p); mm[f"cnn_{exp}"][seed] = m; manifest[str(p)] = file_hash(p)
    data[name], models[name] = d, mm
    print(f"{name}: held-out {sorted(heldout)} ({len(d['X_zd']):,} records); seeds {meta['seeds']}; "
          f"H3 synthetic reserve {len(d['h3_synth']):,}; partition overlaps {overlaps[name]}")
save_json(OUT / "input_manifest.json", manifest)
save_json(OUT / "overlap_audit.json", overlaps)

CIC-IDS2017: held-out ['Web Attack � Sql Injection'] (21 records); seeds [42, 43, 44]; H3 synthetic reserve 49,046; partition overlaps {'train_standard': 0, 'train_heldout': 0, 'standard_heldout': 0}
UNSW-NB15: held-out ['Shellcode'] (1,446 records); seeds [42, 43, 44]; H3 synthetic reserve 3,136; partition overlaps {'train_standard': 0, 'train_heldout': 0, 'standard_heldout': 0}
NSL-KDD: held-out ['buffer_overflow', 'loadmodule', 'perl', 'rootkit'] (88 records); seeds [42, 43, 44]; H3 synthetic reserve 2,874; partition overlaps {'train_standard': 0, 'train_heldout': 0, 'standard_heldout': 0}


---
## Step 7 · Score every model and seed
For each run, class probabilities are computed once for the validation, standard test and held-out sets; every metric below is derived from these same probabilities.

In [5]:
# ────────────────────────────────────────────────────────────────────────────
# PROBABILITIES AND PER-RUN METRICS
#
# For each (dataset, model, seed):
#   attack score s = 1 − P(benign)
#   threshold t = value above which FPR_TARGET of validation benign records fall
#   known macro-F1      argmax predictions on every standard-test record
#                       (a Synthetic-OOD prediction on a known record is an error)
#   detection recall    share of held-out records with s > t
#   benign FAR (test)   share of standard-test benign records with s > t
#   detection F1        held-out records positive, standard-test benign negative, at t
#   AUROC               held-out vs standard-test benign, threshold-free
#   argmax recall/FAR   the same using "top prediction is not benign" (no threshold)
#   Synthetic-OOD rate  (B only) held-out records whose top prediction is Synthetic-OOD,
#                       and the false-OOD rate on known-class test records
# ────────────────────────────────────────────────────────────────────────────
def probs(model, key, X):
    if key == "rf":
        return model.predict_proba(X).astype(np.float32)
    return model.predict(X[..., None], batch_size=4096, verbose=0).astype(np.float32)

def threshold_at(benign_scores, fpr):
    return float(np.quantile(benign_scores, 1.0 - fpr, method="higher"))

def det_f1(tp, fn, fp):
    return 2 * tp / (2 * tp + fn + fp) if (2 * tp + fn + fp) else 0.0

P, run_rows = {}, []
for name in DATASETS:
    d, ben = data[name], data[name]["benign_id"]
    ood = len(d["classes_a"])
    val_ben = d["y_val"] == ben
    std_ben = d["y_std"] == ben
    P[name] = {}
    for key in ["rf", "cnn_A", "cnn_B"]:
        P[name][key] = {}
        for seed, model in models[name][key].items():
            pv, ps, pz = probs(model, key, d["X_val"]), probs(model, key, d["X_std"]), probs(model, key, d["X_zd"])
            s_cal, s_std, s_zd = 1 - pv[val_ben, ben], 1 - ps[:, ben], 1 - pz[:, ben]
            yp_val, yp_std, yp_zd = pv.argmax(1), ps.argmax(1), pz.argmax(1)
            r = dict(dataset=name, model=key, seed=seed,
                     val_macro_f1=float(f1_score(d["y_val"], yp_val, labels=np.unique(d["y_val"]), average="macro", zero_division=0)),
                     known_macro_f1=float(f1_score(d["y_std"], yp_std, labels=np.unique(d["y_std"]), average="macro", zero_division=0)),
                     known_accuracy=float(accuracy_score(d["y_std"], yp_std)),
                     auroc=float(roc_auc_score(np.r_[np.zeros(std_ben.sum()), np.ones(len(s_zd))], np.r_[s_std[std_ben], s_zd])),
                     argmax_recall=float((yp_zd != ben).mean()), argmax_benign_far=float((yp_std[std_ben] != ben).mean()),
                     exact_heldout_label_recall=0.0)
            for fpr in [FPR_TARGET] + FPR_SENSITIVITY:
                t = threshold_at(s_cal, fpr)
                tp = int((s_zd > t).sum()); fp = int((s_std[std_ben] > t).sum())
                sfx = "" if fpr == FPR_TARGET else f"_at_{fpr:g}"
                r.update({f"threshold{sfx}": t, f"recall{sfx}": tp / len(s_zd),
                          f"benign_far{sfx}": fp / std_ben.sum(),
                          f"detection_f1{sfx}": det_f1(tp, len(s_zd) - tp, fp)})
            r["gap_G"] = r["known_macro_f1"] - r["detection_f1"]
            if key == "cnn_B":
                known_rows = d["y_std"] != ben
                r.update(ood_rate_heldout=float((yp_zd == ood).mean()), ood_count_heldout=int((yp_zd == ood).sum()),
                         false_ood_rate_known=float((yp_std == ood).mean()),
                         false_ood_rate_known_attacks=float((yp_std[known_rows] == ood).mean()) if known_rows.any() else np.nan)
            run_rows.append(r)
            P[name][key][seed] = dict(s_cal=s_cal, s_std=s_std, s_zd=s_zd, yp_std=yp_std, yp_zd=yp_zd,
                                      **({"pood_std": ps[:, ood], "pood_zd": pz[:, ood]} if key == "cnn_B" else {}))
            del pv, ps, pz
        gc.collect()

runs = pd.DataFrame(run_rows)
runs["detection_definition"] = DETECTION_DEFINITION
runs.to_csv(OUT / "per_run_metrics.csv", index=False)

# Run used for paired tests: highest validation macro-F1 per (dataset, model)
SELECTED = {(n, k): int(runs[(runs.dataset == n) & (runs.model == k)].sort_values("val_macro_f1").iloc[-1]["seed"])
            for n in DATASETS for k in ["rf", "cnn_A", "cnn_B"]}
save_json(OUT / "selected_runs.json", {f"{n}|{k}": s for (n, k), s in SELECTED.items()})

cols = ["known_macro_f1", "known_accuracy", "recall", "benign_far", "detection_f1", "gap_G", "auroc", "argmax_recall", "argmax_benign_far"]
seed_summary = runs.groupby(["dataset", "model"])[cols].agg(["mean", "std"])
seed_summary.to_csv(OUT / "seed_summary_mean_sd.csv")
with pd.option_context("display.float_format", "{:.4f}".format):
    display(seed_summary)

known_macro_f1        known_accuracy        recall         \
                            mean    std           mean    std   mean    std   
dataset     model                                                             
CIC-IDS2017 cnn_A         0.3955 0.0387         0.8379 0.0452 0.0159 0.0275   
            cnn_B         0.4057 0.0749         0.8133 0.0879 0.0000 0.0000   
            rf            0.8946 0.0076         0.9988 0.0000 1.0000 0.0000   
NSL-KDD     cnn_A         0.3455 0.0428         0.5599 0.1578 0.1439 0.1576   
            cnn_B         0.3914 0.0041         0.5246 0.0103 0.0795 0.0745   
            rf            0.7451 0.0047         0.9954 0.0001 0.8409 0.0114   
UNSW-NB15   cnn_A         0.4290 0.0017         0.6779 0.0027 0.1521 0.2062   
            cnn_B         0.4190 0.0149         0.6801 0.0051 0.1621 0.0915   
            rf            0.6505 0.0055         0.8785 0.0007 0.5526 0.0025   

                  benign_far        detection_f1          gap_G         auroc  \
                        mean    std         mean    std    mean    std   mean   
dataset     model                                                               
CIC-IDS2017 cnn_A     0.0099 0.0003       0.0002 0.0004  0.3953 0.0387 0.8981   
            cnn_B     0.0099 0.0000       0.0000 0.0000  0.4057 0.0749 0.9186   
            rf        0.0079 0.0001       0.0175 0.0003  0.8772 0.0073 0.9992   
NSL-KDD     cnn_A     0.0121 0.0016       0.1069 0.1191  0.2385 0.0784 0.9462   
            cnn_B     0.0115 0.0002       0.0606 0.0554  0.3308 0.0565 0.9526   
            rf        0.0096 0.0004       0.5436 0.0140  0.2015 0.0168 0.9937   
UNSW-NB15   cnn_A     0.0094 0.0012       0.2145 0.2722  0.2145 0.2710 0.9386   
            cnn_B     0.0096 0.0004       0.2535 0.1339  0.1655 0.1392 0.9312   
            rf        0.0078 0.0002       0.6815 0.0027 -0.0311 0.0035 0.9806   

                         argmax_recall        argmax_benign_far         
                     std          mean    std              mean    std  
dataset     model                                                       
CIC-IDS2017 cnn_A 0.0326        0.9683 0.0550            0.1900 0.0509  
            cnn_B 0.0248        0.9683 0.0550            0.2207 0.1020  
            rf    0.0001        0.4444 0.0275            0.0004 0.0000  
NSL-KDD     cnn_A 0.0177        1.0000 0.0000            0.7030 0.2807  
            cnn_B 0.0052        1.0000 0.0000            0.8088 0.0195  
            rf    0.0002        0.0492 0.0066            0.0018 0.0001  
UNSW-NB15   cnn_A 0.0136        1.0000 0.0000            0.2843 0.0009  
            cnn_B 0.0074        0.9998 0.0004            0.2865 0.0052  
            rf    0.0003        0.8029 0.0030            0.0560 0.0004

In [6]:
# ────────────────────────────────────────────────────────────────────────────
# DESCRIPTIVE: SYNTHETIC-OOD OUTPUT OF EXPERIMENT B, AND THE EXACT-LABEL CHECK
# Not part of any hypothesis test. Counts are per seed.
# ────────────────────────────────────────────────────────────────────────────
ood_table = runs[runs.model == "cnn_B"][["dataset", "seed", "ood_count_heldout", "ood_rate_heldout",
                                         "false_ood_rate_known", "false_ood_rate_known_attacks"]]
ood_table.to_csv(OUT / "synthetic_ood_descriptive.csv", index=False)
display(ood_table)
print("Exact held-out label recall is 0 for every model by construction: no model has an output "
      "for the held-out class, so this definition cannot distinguish models and is not used.")

,dataset,seed,ood_count_heldout,ood_rate_heldout,false_ood_rate_known,false_ood_rate_known_attacks
6,CIC-IDS2017,42,0.0,0.000000,0.036110,0.001333
7,CIC-IDS2017,43,3.0,0.142857,0.026854,0.000188
8,CIC-IDS2017,44,0.0,0.000000,0.009118,0.000157
15,UNSW-NB15,42,33.0,0.022822,0.002894,0.004041
16,UNSW-NB15,43,45.0,0.031120,0.005031,0.006736
17,UNSW-NB15,44,31.0,0.021438,0.009172,0.006114
24,NSL-KDD,42,0.0,0.000000,0.015837,0.004127
25,NSL-KDD,43,0.0,0.000000,0.025330,0.004990
26,NSL-KDD,44,0.0,0.000000,0.028798,0.004607


Exact held-out label recall is 0 for every model by construction: no model has an output for the held-out class, so this definition cannot distinguish models and is not used.


### Detection counts: known and unknown attacks per model
How many attack records each model flagged, at the same 1% validation false-alarm threshold used for H1 and H2.

- **Known attacks:** attack records in the standard test set (classes seen in training). *Detected* = flagged as an attack; *correct class* = also given the right attack label.
- **Unknown attacks:** the held-out (zero-day) records. *Detected* = flagged as an attack; the exact class cannot be named by any model.
- **Benign false alarms:** benign test records flagged, for context.

The first table is the run used for the paired tests (best validation macro-F1); the second is the mean ± SD over the three seeds.

In [7]:
# ────────────────────────────────────────────────────────────────────────────
# DETECTION COUNTS TABLE
#
# For each dataset, model and seed, counts at the FPR_TARGET threshold:
#   known attacks   = standard-test records whose true class is not benign
#   detected        = attack score above the threshold
#   correct class   = detected AND top prediction equals the true class
#   unknown attacks = held-out records; detected = attack score above the threshold
# Saved as detection_counts_by_seed.csv and detection_counts_selected.csv.
# ────────────────────────────────────────────────────────────────────────────
count_rows = []
for name in DATASETS:
    d, ben = data[name], data[name]["benign_id"]
    known_att = d["y_std"] != ben
    for key in ["rf", "cnn_A", "cnn_B"]:
        for seed, q in P[name][key].items():
            t = threshold_at(q["s_cal"], FPR_TARGET)
            flag_std, flag_zd = q["s_std"] > t, q["s_zd"] > t
            n_known, n_unknown, n_ben = int(known_att.sum()), len(flag_zd), int((~known_att).sum())
            det_known = int(flag_std[known_att].sum())
            correct = int((flag_std & (q["yp_std"] == d["y_std"]))[known_att].sum())
            det_unknown = int(flag_zd.sum())
            count_rows.append({
                "dataset": name, "model": MODEL_NAMES[key], "seed": seed,
                "selected": seed == SELECTED[(name, key)],
                "known attacks": n_known, "known detected": det_known,
                "known detected %": 100 * det_known / max(n_known, 1),
                "known correct class": correct, "known correct class %": 100 * correct / max(n_known, 1),
                "known missed": n_known - det_known,
                "unknown attacks": n_unknown, "unknown detected": det_unknown,
                "unknown detected %": 100 * det_unknown / max(n_unknown, 1),
                "unknown missed": n_unknown - det_unknown,
                "benign records": n_ben, "benign false alarms": int(flag_std[~known_att].sum()),
            })
counts = pd.DataFrame(count_rows)
counts.to_csv(OUT / "detection_counts_by_seed.csv", index=False)

count_table = counts[counts["selected"]].drop(columns=["selected"]).reset_index(drop=True)
count_table.to_csv(OUT / "detection_counts_selected.csv", index=False)
with pd.option_context("display.float_format", "{:.1f}".format, "display.max_columns", 30, "display.width", 250):
    print(f"Selected runs — counts at the {FPR_TARGET:.0%} validation false-alarm threshold")
    display(count_table)
    pct_cols = ["known detected %", "known correct class %", "unknown detected %", "benign false alarms"]
    print("Mean ± SD over seeds")
    display(counts.groupby(["dataset", "model"])[pct_cols].agg(["mean", "std"]))

Selected runs — counts at the 1% validation false-alarm threshold


,dataset,model,seed,known attacks,known detected,known detected %,known correct class,known correct class %,known missed,unknown attacks,unknown detected,unknown detected %,unknown missed,benign records,benign false alarms
0,CIC-IDS2017,Random Forest (A),43,63771,63767,100.0,63444,99.5,4,21,21,100.0,0,299560,2318
1,CIC-IDS2017,CNN-LSTM A (standard),42,63771,47369,74.3,46848,73.5,16402,21,0,0.0,21,299560,2879
2,CIC-IDS2017,CNN-LSTM B (augmented),44,63771,47283,74.1,47071,73.8,16488,21,0,0.0,21,299560,2947
3,UNSW-NB15,Random Forest (A),42,9650,7222,74.8,6421,66.5,2428,1446,796,55.0,650,12809,101
4,UNSW-NB15,CNN-LSTM A (standard),44,9650,4587,47.5,3037,31.5,5063,1446,91,6.3,1355,12809,121
5,UNSW-NB15,CNN-LSTM B (augmented),42,9650,4571,47.4,2962,30.7,5079,1446,82,5.7,1364,12809,125
6,NSL-KDD,Random Forest (A),43,10420,10405,99.9,10340,99.2,15,88,73,83.0,15,11491,112
7,NSL-KDD,CNN-LSTM A (standard),42,10420,6940,66.6,6397,61.4,3480,88,28,31.8,60,11491,118
8,NSL-KDD,CNN-LSTM B (augmented),42,10420,7471,71.7,6903,66.2,2949,88,6,6.8,82,11491,130


Mean ± SD over seeds


known detected %      known correct class %      unknown detected %      benign false alarms     
                                               mean  std                  mean  std               mean  std                mean  std
dataset     model                                                                                                                   
CIC-IDS2017 CNN-LSTM A (standard)              71.3  4.4                  69.9  4.8                1.6  2.7              2951.7 75.7
            CNN-LSTM B (augmented)             66.5 11.8                  65.5 12.9                0.0  0.0              2951.3  8.4
            Random Forest (A)                 100.0  0.0                  99.5  0.0              100.0  0.0              2365.0 42.0
NSL-KDD     CNN-LSTM A (standard)              74.5  6.9                  66.0  4.1               14.4 15.8               139.3 18.5
            CNN-LSTM B (augmented)             73.4  1.5                  67.2  0.8                8.0  7.5               132.0  1.7
            Random Forest (A)                  99.9  0.0                  99.2  0.0               84.1  1.1               110.3  4.7
UNSW-NB15   CNN-LSTM A (standard)              45.4  3.7                  29.5  3.5               15.2 20.6               120.7 15.5
            CNN-LSTM B (augmented)             44.0  5.4                  27.7  5.0               16.2  9.2               122.3  4.6
            Random Forest (A)                  75.0  0.1                  66.7  0.2               55.3  0.2                99.7  2.3

---
## H1 · Generalization Gap — paired bootstrap at a common false-alarm rate (Section 3.10.1)
G = F1_known − F1_zero-day, where F1_zero-day is detection F1 at the validation-calibrated 1% false-alarm threshold. On each of `N_BOOT` replicates the same resampled test records are scored for CNN-LSTM A and B. **SUPPORTED** requires all three: the upper bound for ΔG = G_B − G_A is below 0; the lower bound for Δ detection F1 is above 0; the lower bound for Δ known macro-F1 is above −0.02. Intervals are at the 1 − 0.05/3 level. The test is run on the validation-selected runs and, as a robustness check, for every seed with A and B paired by seed.

In [8]:
# ────────────────────────────────────────────────────────────────────────────
# H1 PAIRED BOOTSTRAP
# Run once on the validation-selected runs (the reported test) and once for
# every seed, pairing CNN-LSTM A and B trained with the same seed, so the
# per-seed intervals sit next to the selected result.
# Macro-F1 is recomputed from a weighted confusion matrix, so each replicate
# costs one bincount per model rather than a full re-scoring.
# ────────────────────────────────────────────────────────────────────────────
def macro_f1_weighted(codes, weights, labels, k):
    cm = np.bincount(codes, weights=weights, minlength=k * k).reshape(k, k).astype(np.float64)
    tp = cm[labels, labels]
    den = cm[labels, :].sum(1) + cm[:, labels].sum(0)
    return float(np.mean(np.divide(2 * tp, den, out=np.zeros_like(tp), where=den > 0)))

def h1_bootstrap(name, seed_a, seed_b, n_boot=N_BOOT):
    d, ben = data[name], data[name]["benign_id"]
    k = len(d["classes_b"])
    labels = np.unique(d["y_std"])
    std_ben = d["y_std"] == ben
    runs_ab = {}
    for key, seed in [("cnn_A", seed_a), ("cnn_B", seed_b)]:
        q = P[name][key][seed]
        t = threshold_at(q["s_cal"], FPR_TARGET)
        runs_ab[key] = dict(codes=d["y_std"] * k + q["yp_std"], alert_zd=q["s_zd"] > t, alert_ben=q["s_std"][std_ben] > t)
    rng = np.random.default_rng(42)
    n_s, n_z = len(d["y_std"]), len(d["X_zd"])
    point, draws = {}, {"d_gap": [], "d_det": [], "d_known": []}
    for b in range(n_boot + 1):
        ws = np.ones(n_s) if b == 0 else np.bincount(rng.integers(0, n_s, n_s), minlength=n_s).astype(float)
        wz = np.ones(n_z) if b == 0 else np.bincount(rng.integers(0, n_z, n_z), minlength=n_z).astype(float)
        vals = {}
        for key, r in runs_ab.items():
            fk = macro_f1_weighted(r["codes"], ws, labels, k)
            tp = wz[r["alert_zd"]].sum(); fp = ws[std_ben][r["alert_ben"]].sum()
            vals[key] = (fk, det_f1(tp, wz.sum() - tp, fp))
        (fka, fda), (fkb, fdb) = vals["cnn_A"], vals["cnn_B"]
        if b == 0:
            point = dict(F1_known_A=fka, F1_known_B=fkb, F1_zd_A=fda, F1_zd_B=fdb, G_A=fka - fda, G_B=fkb - fdb)
            continue
        draws["d_gap"].append((fkb - fdb) - (fka - fda)); draws["d_det"].append(fdb - fda); draws["d_known"].append(fkb - fka)
    ci = {kk: np.percentile(v, [100 * ALPHA / 2, 100 * (1 - ALPHA / 2)]).tolist() for kk, v in draws.items()}
    cond = dict(gap_falls=ci["d_gap"][1] < 0, detection_rises=ci["d_det"][0] > 0, known_within_margin=ci["d_known"][0] > -KNOWN_F1_MARGIN)
    if all(cond.values()):
        verdict = "SUPPORTED"
    elif cond["gap_falls"] and cond["detection_rises"]:
        verdict = "DETECTION GAIN; KNOWN-CLASS LOSS EXCEEDS MARGIN"
    elif cond["gap_falls"]:
        verdict = "GAP FROM KNOWN-CLASS LOSS"
    elif point["G_B"] < point["G_A"]:
        verdict = "DIRECTIONAL ONLY"
    else:
        verdict = "NOT SUPPORTED"
    return dict(verdict=verdict, **point, delta_G=point["G_B"] - point["G_A"], ci=ci, conditions=cond)

h1, h1_seed_rows = {}, []
for name in DATASETS:
    r = h1_bootstrap(name, SELECTED[(name, "cnn_A")], SELECTED[(name, "cnn_B")])
    h1[name] = dict(r, seeds=dict(A=SELECTED[(name, "cnn_A")], B=SELECTED[(name, "cnn_B")]),
                    fpr_target=FPR_TARGET, ci_level=1 - ALPHA, replicates=N_BOOT, known_f1_margin=KNOWN_F1_MARGIN,
                    detection_definition=DETECTION_DEFINITION)
    for seed in data[name]["seeds"]:
        rs = h1_bootstrap(name, seed, seed)
        h1_seed_rows.append(dict(dataset=name, seed=seed, G_A=rs["G_A"], G_B=rs["G_B"], dG=rs["delta_G"], dG_CI=rs["ci"]["d_gap"],
                                 dF1_known=rs["F1_known_B"] - rs["F1_known_A"], dF1_known_CI=rs["ci"]["d_known"],
                                 dF1_zd=rs["F1_zd_B"] - rs["F1_zd_A"], dF1_zd_CI=rs["ci"]["d_det"], verdict=rs["verdict"]))
save_json(OUT / "h1_results.json", h1)
h1_table = pd.DataFrame([dict(dataset=n, G_A=v["G_A"], G_B=v["G_B"], dF1_known=v["F1_known_B"] - v["F1_known_A"],
                              dF1_known_CI=v["ci"]["d_known"], dF1_zd=v["F1_zd_B"] - v["F1_zd_A"], dF1_zd_CI=v["ci"]["d_det"],
                              dG=v["delta_G"], dG_CI=v["ci"]["d_gap"], verdict=v["verdict"]) for n, v in h1.items()])
h1_seeds = pd.DataFrame(h1_seed_rows)
h1_seeds.to_csv(OUT / "h1_by_seed.csv", index=False)
with pd.option_context("display.float_format", "{:.4f}".format, "display.max_colwidth", 60):
    print("Validation-selected runs (reported test)"); display(h1_table)
    print("Every seed, A and B paired by seed"); display(h1_seeds)

Validation-selected runs (reported test)


,dataset,G_A,G_B,dF1_known,dF1_known_CI,dF1_zd,dF1_zd_CI,dG,dG_CI,verdict
0,CIC-IDS2017,0.4330,0.4576,0.0246,"[0.009997244797943085, 0.0435790000303983]",0.0000,"[0.0, 0.0]",0.0246,"[0.009997244797943085, 0.0435790000303983]",NOT SUPPORTED
1,UNSW-NB15,0.3200,0.3261,-0.0044,"[-0.013665981990848414, 0.005208923272333639]",-0.0106,"[-0.04616345157872199, 0.026717835028245034]",0.0061,"[-0.03201205182753898, 0.04287620244723656]",NOT SUPPORTED
2,NSL-KDD,0.1556,0.3425,0.0012,"[-0.01383147722083089, 0.01498457161178101]",-0.1857,"[-0.27030773628025095, -0.10173904712002427]",0.1870,"[0.10240948488984325, 0.2713659325713366]",NOT SUPPORTED


Every seed, A and B paired by seed


,dataset,seed,G_A,G_B,dG,dG_CI,dF1_known,dF1_known_CI,dF1_zd,dF1_zd_CI,verdict
0,CIC-IDS2017,42,0.4330,0.3199,-0.1131,"[-0.11746891714202776, -0.10706506262207863]",-0.1131,"[-0.11746891714202776, -0.10706506262207863]",0.0000,"[0.0, 0.0]",GAP FROM KNOWN-CLASS LOSS
1,CIC-IDS2017,43,0.3971,0.4397,0.0426,"[0.03152654873324074, 0.05388145340239305]",0.0419,"[0.030758900328658745, 0.05303976196765629]",-0.0007,"[-0.0026321938629103037, 0.0]",NOT SUPPORTED
2,CIC-IDS2017,44,0.3556,0.4576,0.1020,"[0.08276229571691429, 0.11614880128046212]",0.1020,"[0.08276229571691429, 0.11614880128046212]",0.0000,"[0.0, 0.0]",NOT SUPPORTED
3,UNSW-NB15,42,0.4168,0.3261,-0.0907,"[-0.11748043445530643, -0.06428486776149869]",-0.0017,"[-0.011324921498142106, 0.006320943576578641]",0.0890,"[0.06362441988654276, 0.1143666655431016]",SUPPORTED
4,UNSW-NB15,43,-0.0933,0.0904,0.1837,"[0.15371098499817873, 0.21434059888243912]",-0.0005,"[-0.008854763818708833, 0.008609337562011043]",-0.1842,"[-0.2128216290681513, -0.15478201553102341]",NOT SUPPORTED
5,UNSW-NB15,44,0.3200,0.0800,-0.2400,"[-0.2776969705339417, -0.19839109267661095]",-0.0277,"[-0.03637600693335737, -0.02016784938976783]",0.2123,"[0.17093386214261727, 0.25030443424293475]",DETECTION GAIN; KNOWN-CLASS LOSS EXCEEDS MARGIN
6,NSL-KDD,42,0.1556,0.3425,0.1870,"[0.10240948488984325, 0.2713659325713366]",0.0012,"[-0.01383147722083089, 0.01498457161178101]",-0.1857,"[-0.27030773628025095, -0.10173904712002427]",NOT SUPPORTED
7,NSL-KDD,43,0.3114,0.3806,0.0692,"[0.034426238922212346, 0.10200615996154289]",0.0698,"[0.05945024187650166, 0.08037509383820768]",0.0007,"[-0.03160500860272027, 0.03465539452495975]",NOT SUPPORTED
8,NSL-KDD,44,0.2487,0.2694,0.0207,"[-0.060579484509320075, 0.10026707692609144]",0.0667,"[0.05581668030363638, 0.07700435940883986]",0.0460,"[-0.03355772379491364, 0.12857228930088477]",NOT SUPPORTED


---
## Step 8 · H2 — held-out detection recall (Section 3.10.2)
Primary: recall at the common 1% validation false-alarm threshold. Direction: CNN-LSTM B must exceed **both** Random Forest and CNN-LSTM A. Significance: exact two-sided McNemar test on the paired detections of B and A (selected runs), Bonferroni α = 0.05/3. Cohen's h gives the effect size. The argmax version (top prediction not benign) is reported alongside, with its false-alarm rate.

In [9]:
# ────────────────────────────────────────────────────────────────────────────
# H2 TESTS
# ────────────────────────────────────────────────────────────────────────────
h2 = {}
for name in DATASETS:
    det = {}
    for key in ["rf", "cnn_A", "cnn_B"]:
        q = P[name][key][SELECTED[(name, key)]]
        det[key] = q["s_zd"] > threshold_at(q["s_cal"], FPR_TARGET)
    rec = {k: float(v.mean()) for k, v in det.items()}
    b_only = int((det["cnn_B"] & ~det["cnn_A"]).sum()); a_only = int((det["cnn_A"] & ~det["cnn_B"]).sum())
    p = float(binomtest(b_only, b_only + a_only, 0.5).pvalue) if b_only + a_only else 1.0
    direction = rec["cnn_B"] > rec["cnn_A"] and rec["cnn_B"] > rec["rf"]
    sel = runs.set_index(["dataset", "model", "seed"])
    h2[name] = dict(recall_rf=rec["rf"], recall_A=rec["cnn_A"], recall_B=rec["cnn_B"],
                    benign_far={k: float(sel.loc[(name, k, SELECTED[(name, k)]), "benign_far"]) for k in det},
                    argmax_recall={k: float(sel.loc[(name, k, SELECTED[(name, k)]), "argmax_recall"]) for k in det},
                    argmax_benign_far={k: float(sel.loc[(name, k, SELECTED[(name, k)]), "argmax_benign_far"]) for k in det},
                    B_only=b_only, A_only=a_only, mcnemar_p=p, alpha=ALPHA,
                    cohen_h=float(2 * np.arcsin(np.sqrt(rec["cnn_B"])) - 2 * np.arcsin(np.sqrt(rec["cnn_A"]))),
                    beats_both=direction,
                    verdict="SUPPORTED" if direction and p < ALPHA else ("DIRECTIONAL ONLY" if direction else "NOT SUPPORTED"),
                    fpr_target=FPR_TARGET, detection_definition=DETECTION_DEFINITION)
save_json(OUT / "h2_results.json", h2)
h2_table = pd.DataFrame([dict(dataset=n, recall_RF=v["recall_rf"], recall_A=v["recall_A"], recall_B=v["recall_B"],
                              B_only=v["B_only"], A_only=v["A_only"], mcnemar_p=v["mcnemar_p"], cohen_h=v["cohen_h"],
                              verdict=v["verdict"]) for n, v in h2.items()])
display(h2_table)
recall_seeds = runs.pivot_table(index="dataset", columns="model", values=["recall", "argmax_recall", "argmax_benign_far"], aggfunc=["mean", "std"])
recall_seeds.to_csv(OUT / "h2_recall_by_seed_mean_sd.csv")

,dataset,recall_RF,recall_A,recall_B,B_only,A_only,mcnemar_p,cohen_h,verdict
0,CIC-IDS2017,1.000000,0.000000,0.000000,0,0,1.000000,0.000000,NOT SUPPORTED
1,UNSW-NB15,0.550484,0.062932,0.056708,81,90,0.540812,-0.026253,NOT SUPPORTED
2,NSL-KDD,0.829545,0.318182,0.068182,1,23,0.000003,-0.670271,NOT SUPPORTED


---
## Held-out outcomes · how held-out records were classified
Two plain outcomes are reported for the held-out records, with counts, together with the classes those records were assigned to. Neither needs a threshold:

- **Outcome 1 — detected as an attack:** the top prediction is not the benign class.
- **Outcome 2 — flagged as unknown:** the top prediction is Synthetic-OOD (CNN-LSTM B only).

They are shown with the benign false-alarm rate and the false-OOD rate on known test records, because a detection rate means little without them. The Synthetic-OOD output is also scored threshold-free: AUROC of P(Synthetic-OOD) for held-out records against known-class test records (0.5 = chance).

In [10]:
# ────────────────────────────────────────────────────────────────────────────
# OUTCOMES 1 AND 2, ASSIGNMENTS, AND THE SYNTHETIC-OOD AUROC (every seed)
# ────────────────────────────────────────────────────────────────────────────
outcome_rows, assign_rows = [], []
for name in DATASETS:
    d, ben = data[name], data[name]["benign_id"]
    ood = len(d["classes_a"])
    std_ben = d["y_std"] == ben
    n_zd = len(d["X_zd"])
    for key in ["rf", "cnn_A", "cnn_B"]:
        for seed, q in P[name][key].items():
            r = dict(dataset=name, model=MODEL_NAMES[key], seed=seed, selected=seed == SELECTED[(name, key)],
                     heldout_records=n_zd,
                     outcome1_detected=int((q["yp_zd"] != ben).sum()),
                     outcome1_rate=float((q["yp_zd"] != ben).mean()),
                     benign_false_alarm_rate=float((q["yp_std"][std_ben] != ben).mean()))
            if key == "cnn_B":
                r.update(outcome2_unknown=int((q["yp_zd"] == ood).sum()), outcome2_rate=float((q["yp_zd"] == ood).mean()),
                         false_ood_rate_known=float((q["yp_std"] == ood).mean()),
                         ood_auroc_heldout_vs_known=float(roc_auc_score(np.r_[np.zeros(len(q["pood_std"])), np.ones(n_zd)],
                                                                         np.r_[q["pood_std"], q["pood_zd"]])))
            outcome_rows.append(r)
            names_ = np.array(d["classes_b"])[q["yp_zd"]]
            for cls, n in pd.Series(names_).value_counts().items():
                assign_rows.append(dict(dataset=name, model=MODEL_NAMES[key], seed=seed, assigned_to=cls, records=int(n),
                                        share=n / n_zd))
outcomes = pd.DataFrame(outcome_rows)
outcomes.to_csv(OUT / "heldout_outcomes_by_seed.csv", index=False)
assignments = pd.DataFrame(assign_rows)
assignments.to_csv(OUT / "heldout_assignments_by_seed.csv", index=False)
with pd.option_context("display.float_format", "{:.4f}".format, "display.max_columns", 30, "display.width", 250):
    display(outcomes)
    print("Where held-out records went (validation-selected run, top 5 classes per model)")
    sel_assign = assignments[[s == SELECTED[(n, {v: k for k, v in MODEL_NAMES.items()}[m])]
                              for n, m, s in zip(assignments.dataset, assignments.model, assignments.seed)]]
    display(sel_assign.sort_values(["dataset", "model", "records"], ascending=[True, True, False]).groupby(["dataset", "model"]).head(5))

,dataset,model,seed,selected,heldout_records,outcome1_detected,outcome1_rate,benign_false_alarm_rate,outcome2_unknown,outcome2_rate,false_ood_rate_known,ood_auroc_heldout_vs_known
0,CIC-IDS2017,Random Forest (A),42,False,21,9,0.4286,0.0004,NaN,NaN,NaN,NaN
1,CIC-IDS2017,Random Forest (A),43,True,21,9,0.4286,0.0004,NaN,NaN,NaN,NaN
2,CIC-IDS2017,Random Forest (A),44,False,21,10,0.4762,0.0004,NaN,NaN,NaN,NaN
3,CIC-IDS2017,CNN-LSTM A (standard),42,True,21,21,1.0000,0.1515,NaN,NaN,NaN,NaN
4,CIC-IDS2017,CNN-LSTM A (standard),43,False,21,19,0.9048,0.1708,NaN,NaN,NaN,NaN
5,CIC-IDS2017,CNN-LSTM A (standard),44,False,21,21,1.0000,0.2477,NaN,NaN,NaN,NaN
6,CIC-IDS2017,CNN-LSTM B (augmented),42,False,21,21,1.0000,0.3337,0.0000,0.0000,0.0361,0.3901
7,CIC-IDS2017,CNN-LSTM B (augmented),43,False,21,21,1.0000,0.1930,3.0000,0.1429,0.0269,0.3676
8,CIC-IDS2017,CNN-LSTM B (augmented),44,True,21,19,0.9048,0.1354,0.0000,0.0000,0.0091,0.3905
9,UNSW-NB15,Random Forest (A),42,True,1446,1156,0.7994,0.0559,NaN,NaN,NaN,NaN


Where held-out records went (validation-selected run, top 5 classes per model)


,dataset,model,seed,assigned_to,records,share
15,CIC-IDS2017,CNN-LSTM A (standard),42,SSH-Patator,10,0.4762
16,CIC-IDS2017,CNN-LSTM A (standard),42,DoS GoldenEye,10,0.4762
17,CIC-IDS2017,CNN-LSTM A (standard),42,Web Attack � Brute Force,1,0.0476
33,CIC-IDS2017,CNN-LSTM B (augmented),44,SSH-Patator,12,0.5714
34,CIC-IDS2017,CNN-LSTM B (augmented),44,Web Attack � XSS,5,0.2381
35,CIC-IDS2017,CNN-LSTM B (augmented),44,BENIGN,2,0.0952
36,CIC-IDS2017,CNN-LSTM B (augmented),44,DoS GoldenEye,2,0.0952
5,CIC-IDS2017,Random Forest (A),43,BENIGN,12,0.5714
6,CIC-IDS2017,Random Forest (A),43,Web Attack � Brute Force,5,0.2381
7,CIC-IDS2017,Random Forest (A),43,DoS GoldenEye,2,0.0952


In [11]:
# ────────────────────────────────────────────────────────────────────────────
# PAIRED TESTS FOR EVERY SEED
# Exact McNemar test and Cohen's h on the same records and the same basis (one
# run per model, paired by seed): CNN-LSTM B vs A and B vs Random Forest, for
# detection at the 1% validation threshold and for outcome 1 (argmax).
# ────────────────────────────────────────────────────────────────────────────
def mcnemar_exact(a, b):
    n10, n01 = int((a & ~b).sum()), int((~a & b).sum())
    p = float(binomtest(n10, n10 + n01, 0.5).pvalue) if n10 + n01 else 1.0
    return n10, n01, p

def cohen_h(p1, p2):
    return float(2 * np.arcsin(np.sqrt(p1)) - 2 * np.arcsin(np.sqrt(p2)))

pair_rows = []
for name in DATASETS:
    d, ben = data[name], data[name]["benign_id"]
    for seed in d["seeds"]:
        det = {}
        for key in ["rf", "cnn_A", "cnn_B"]:
            q = P[name][key][seed]
            det[(key, "threshold_1pct")] = q["s_zd"] > threshold_at(q["s_cal"], FPR_TARGET)
            det[(key, "argmax_not_benign")] = q["yp_zd"] != ben
        for rule in ["threshold_1pct", "argmax_not_benign"]:
            for other in ["cnn_A", "rf"]:
                a, b = det[("cnn_B", rule)], det[(other, rule)]
                n10, n01, p = mcnemar_exact(a, b)
                pair_rows.append(dict(dataset=name, seed=seed, rule=rule, comparison=f"B vs {MODEL_NAMES[other]}",
                                      recall_B=a.mean(), recall_other=b.mean(), B_only=n10, other_only=n01,
                                      mcnemar_p=p, significant=p < ALPHA, cohen_h=cohen_h(a.mean(), b.mean())))
pairs = pd.DataFrame(pair_rows)
pairs.to_csv(OUT / "paired_tests_by_seed.csv", index=False)
with pd.option_context("display.float_format", "{:.4f}".format, "display.width", 250):
    display(pairs)

,dataset,seed,rule,comparison,recall_B,recall_other,B_only,other_only,mcnemar_p,significant,cohen_h
0,CIC-IDS2017,42,threshold_1pct,B vs CNN-LSTM A (standard),0.0000,0.0000,0,0,1.0000,False,0.0000
1,CIC-IDS2017,42,threshold_1pct,B vs Random Forest (A),0.0000,1.0000,0,21,0.0000,True,-3.1416
2,CIC-IDS2017,42,argmax_not_benign,B vs CNN-LSTM A (standard),1.0000,1.0000,0,0,1.0000,False,0.0000
3,CIC-IDS2017,42,argmax_not_benign,B vs Random Forest (A),1.0000,0.4286,12,0,0.0005,True,1.7141
4,CIC-IDS2017,43,threshold_1pct,B vs CNN-LSTM A (standard),0.0000,0.0476,0,1,1.0000,False,-0.4400
5,CIC-IDS2017,43,threshold_1pct,B vs Random Forest (A),0.0000,1.0000,0,21,0.0000,True,-3.1416
6,CIC-IDS2017,43,argmax_not_benign,B vs CNN-LSTM A (standard),1.0000,0.9048,2,0,0.5000,False,0.6275
7,CIC-IDS2017,43,argmax_not_benign,B vs Random Forest (A),1.0000,0.4286,12,0,0.0005,True,1.7141
8,CIC-IDS2017,44,threshold_1pct,B vs CNN-LSTM A (standard),0.0000,0.0000,0,0,1.0000,False,0.0000
9,CIC-IDS2017,44,threshold_1pct,B vs Random Forest (A),0.0000,1.0000,0,21,0.0000,True,-3.1416


---
## Step 9 · H3 — SHAP ranking similarity with controls (Section 3.10.3)
For each known attack class c and each model (selected run), SHAP values for output c are computed on four sets of records: real test records of class c (two disjoint sets, R1 and R2), reserved synthetic records whose source class is c (S, never used in training), and real benign records (B). Rankings are mean |SHAP| per feature.

- **ρ(R1, S)** — the H3 statistic (criterion ρ > 0.8), with a permutation p-value.
- **ρ(R1, R2)** — ceiling: how similar two samples of the *same* real class are.
- **ρ(R1, B)** — floor: how similar real attacks are to benign traffic on the same output.

ρ(R1, S) is evidence of semantic fidelity only if it is close to the ceiling **and** clearly above the floor. A high ρ(R1, B) means the ranking mostly reflects the model, not the records.

In [12]:
# ────────────────────────────────────────────────────────────────────────────
# SHAP HELPERS
# class_values extracts output `index` from any SHAP layout; rho_perm returns
# Spearman ρ and a permutation p-value, (k + 1) / (n + 1).
# ────────────────────────────────────────────────────────────────────────────
import shap

def class_values(values, n, p, index):
    if isinstance(values, list):
        return np.asarray(values[index]).reshape(n, p)
    v = np.asarray(values)
    return v.reshape(n, p, -1)[:, :, index]

def rho_perm(x, y, n_perm=SHAP_PERMUTATIONS):
    if np.ptp(x) == 0 or np.ptp(y) == 0:
        return np.nan, np.nan
    rho = float(spearmanr(x, y).statistic)
    rng = np.random.default_rng(42)
    null = np.array([spearmanr(x, rng.permutation(y)).statistic for _ in range(n_perm)])
    return rho, float((1 + (np.abs(null) >= abs(rho)).sum()) / (n_perm + 1))

def explain(model, key, X, ci, bg):
    if key == "rf":
        return class_values(shap.TreeExplainer(model).shap_values(X, check_additivity=False), len(X), X.shape[1], ci), "TreeExplainer"
    try:
        ex = shap.GradientExplainer(model, bg[..., None])
        return class_values(ex.shap_values(X[..., None]), len(X), X.shape[1], ci), "GradientExplainer"
    except Exception:
        f = lambda Z: model.predict(Z[..., None], verbose=0)[:, ci]
        return shap.PermutationExplainer(f, bg, seed=42)(X, max_evals=2 * X.shape[1] + 1).values, "PermutationExplainer"

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# RUN H3
# Classes with fewer than 2 × 5 real test records or 5 reserved synthetic
# records are reported as not evaluated. Beeswarm and bar plots are saved for
# CNN-LSTM B.
# ────────────────────────────────────────────────────────────────────────────
h3_rows = []
for name in DATASETS if RUN_SHAP else []:
    d, lab, fc = data[name], data[name]["label_col"], data[name]["feat_cols"]
    ben_name = d["classes_a"][d["benign_id"]]
    rng = np.random.default_rng(42)
    bg = d["train"][fc].sample(n=min(SHAP_BACKGROUND, len(d["train"])), random_state=42).to_numpy(np.float32)
    X_ben_all = d["test_standard"].loc[d["test_standard"][lab] == ben_name, fc].to_numpy(np.float32)
    for c in d["classes_a"]:
        if c == ben_name:
            continue
        real = d["test_standard"].loc[d["test_standard"][lab] == c, fc].to_numpy(np.float32)
        synth = d["h3_synth"].loc[d["h3_synth"]["source_class"] == c, fc].to_numpy(np.float32)
        n = min(SHAP_N, len(real) // 2, len(synth))
        if n < 5:
            h3_rows.append(dict(dataset=name, cls=c, status="not evaluated: too few records",
                                n_real=len(real), n_synth=len(synth)))
            continue
        ri = rng.permutation(len(real))
        sets = dict(R1=real[ri[:n]], R2=real[ri[n:2 * n]],
                    S=synth[rng.choice(len(synth), n, replace=False)],
                    B=X_ben_all[rng.choice(len(X_ben_all), min(n, len(X_ben_all)), replace=False)])
        stacked = np.concatenate(list(sets.values()))
        bounds = np.cumsum([0] + [len(v) for v in sets.values()])
        for key in SHAP_MODELS:
            model = models[name][key][SELECTED[(name, key)]]
            ci = d["code"][c]
            vals, method = explain(model, key, stacked, ci, bg)
            prof = {s: np.abs(vals[bounds[i]:bounds[i + 1]]).mean(0) for i, s in enumerate(sets)}
            rho_s, p_s = rho_perm(prof["R1"], prof["S"])
            rho_c = float(spearmanr(prof["R1"], prof["R2"]).statistic)
            rho_b = float(spearmanr(prof["R1"], prof["B"]).statistic)
            h3_rows.append(dict(dataset=name, cls=c, model=key, status="evaluated", explainer=method, n=n,
                                rho_real_synth=rho_s, p_perm=p_s, rho_real_real=rho_c, rho_real_benign=rho_b,
                                exceeds_08=bool(np.isfinite(rho_s) and rho_s > H3_THRESHOLD),
                                above_benign_floor=bool(np.isfinite(rho_s) and rho_s > rho_b)))
            if key == "cnn_B":
                stem = re.sub(r"[^A-Za-z0-9_-]+", "_", f"{name}_{key}_{c}")
                order = np.argsort(prof["R1"])[::-1]
                for tag, sl in [("real", slice(bounds[0], bounds[1])), ("synthetic", slice(bounds[2], bounds[3]))]:
                    try:
                        shap.plots.beeswarm(shap.Explanation(values=vals[sl], data=stacked[sl], feature_names=fc),
                                            show=False, max_display=15, order=order)
                        plt.title(f"{name} · {c} · {tag}"); plt.savefig(OUT / f"{stem}_{tag}_beeswarm.png", dpi=130, bbox_inches="tight")
                    finally:
                        plt.close("all")
                top = order[:15][::-1]
                fig, axes = plt.subplots(1, 4, figsize=(18, 5), sharey=True)
                for ax, s in zip(axes, ["R1", "R2", "S", "B"]):
                    ax.barh(np.arange(len(top)), prof[s][top]); ax.set_title(s); ax.set_xlabel("mean |SHAP|")
                axes[0].set_yticks(np.arange(len(top)), [fc[i][:28] for i in top])
                fig.suptitle(f"{name} · {c} · ρ(R1,S)={rho_s:.3f}  ρ(R1,R2)={rho_c:.3f}  ρ(R1,B)={rho_b:.3f}")
                fig.tight_layout(); fig.savefig(OUT / f"{stem}_profiles.png", dpi=130); plt.close(fig)
        print(f"{name} · {c}: done", flush=True)

h3 = pd.DataFrame(h3_rows)
h3.to_csv(OUT / "h3_results.csv", index=False)
with pd.option_context("display.float_format", "{:.3f}".format, "display.max_rows", 200):
    display(h3)

CIC-IDS2017 · Bot: done
CIC-IDS2017 · DDoS: done
CIC-IDS2017 · DoS GoldenEye: done
CIC-IDS2017 · DoS Hulk: done
CIC-IDS2017 · DoS Slowhttptest: done
CIC-IDS2017 · DoS slowloris: done
CIC-IDS2017 · FTP-Patator: done
CIC-IDS2017 · PortScan: done
CIC-IDS2017 · SSH-Patator: done
CIC-IDS2017 · Web Attack � Brute Force: done
CIC-IDS2017 · Web Attack � XSS: done
UNSW-NB15 · Analysis: done
UNSW-NB15 · Backdoor: done
UNSW-NB15 · DoS: done
UNSW-NB15 · Exploits: done
UNSW-NB15 · Fuzzers: done
UNSW-NB15 · Generic: done
UNSW-NB15 · Reconnaissance: done
UNSW-NB15 · Worms: done
NSL-KDD · apache2: done
NSL-KDD · back: done
NSL-KDD · httptunnel: done
NSL-KDD · ipsweep: done
NSL-KDD · mailbomb: done
NSL-KDD · neptune: done
NSL-KDD · nmap: done
NSL-KDD · pod: done
NSL-KDD · portsweep: done
NSL-KDD · processtable: done
NSL-KDD · saint: done
NSL-KDD · satan: done
NSL-KDD · smurf: done
NSL-KDD · snmpgetattack: done
NSL-KDD · snmpguess: done
NSL-KDD · teardrop: done
NSL-KDD · warezclient: done
NSL-KDD · ware

,dataset,cls,model,status,explainer,n,rho_real_synth,p_perm,rho_real_real,rho_real_benign,exceeds_08,above_benign_floor,n_real,n_synth
0,CIC-IDS2017,Bot,cnn_B,evaluated,GradientExplainer,30.000,0.943,0.001,0.989,0.883,True,True,NaN,NaN
1,CIC-IDS2017,Bot,cnn_A,evaluated,GradientExplainer,30.000,0.948,0.001,0.996,0.957,True,False,NaN,NaN
2,CIC-IDS2017,Bot,rf,evaluated,TreeExplainer,30.000,0.788,0.001,0.987,0.841,False,False,NaN,NaN
3,CIC-IDS2017,DDoS,cnn_B,evaluated,GradientExplainer,30.000,0.894,0.001,0.996,0.891,True,True,NaN,NaN
4,CIC-IDS2017,DDoS,cnn_A,evaluated,GradientExplainer,30.000,0.919,0.001,0.994,0.911,True,True,NaN,NaN
5,CIC-IDS2017,DDoS,rf,evaluated,TreeExplainer,30.000,0.846,0.001,0.998,0.928,True,False,NaN,NaN
6,CIC-IDS2017,DoS GoldenEye,cnn_B,evaluated,GradientExplainer,24.000,0.740,0.001,0.963,0.790,False,False,NaN,NaN
7,CIC-IDS2017,DoS GoldenEye,cnn_A,evaluated,GradientExplainer,24.000,0.815,0.001,0.969,0.846,True,False,NaN,NaN
8,CIC-IDS2017,DoS GoldenEye,rf,evaluated,TreeExplainer,24.000,0.499,0.002,0.963,0.842,False,False,NaN,NaN
9,CIC-IDS2017,DoS Hulk,cnn_B,evaluated,GradientExplainer,30.000,0.809,0.001,0.986,0.779,True,True,NaN,NaN


---
## Summary and report

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# SUMMARY TABLE, JSON AND HTML REPORT
# Every view reads the verdicts computed above; nothing is recomputed here.
# ────────────────────────────────────────────────────────────────────────────
summary = []
for name in DATASETS:
    ev = h3[(h3.dataset == name) & (h3["model"] == "cnn_B") & (h3.status == "evaluated")] if "model" in h3 else pd.DataFrame()
    summary.append(dict(dataset=name, H1=h1[name]["verdict"], dG=h1[name]["delta_G"],
                        H2=h2[name]["verdict"], recall_B=h2[name]["recall_B"], recall_A=h2[name]["recall_A"],
                        recall_RF=h2[name]["recall_rf"], mcnemar_p=h2[name]["mcnemar_p"],
                        H3_classes=len(ev), H3_above_08=int(ev["exceeds_08"].sum()) if len(ev) else 0,
                        H3_above_benign_floor=int(ev["above_benign_floor"].sum()) if len(ev) else 0))
summary = pd.DataFrame(summary)
summary.to_csv(OUT / "evaluation_summary.csv", index=False)
display(summary)

payload = dict(evaluation_id=EVAL_ID, detection_definition=DETECTION_DEFINITION, fpr_target=FPR_TARGET,
               selected_runs={f"{n}|{k}": s for (n, k), s in SELECTED.items()},
               H1=h1, H2=h2, H3=h3.to_dict(orient="records"), summary=summary.to_dict(orient="records"),
               per_run=runs.to_dict(orient="records"), overlaps=overlaps,
               h1_by_seed=h1_seeds.to_dict(orient="records"), heldout_outcomes=outcomes.to_dict(orient="records"),
               paired_tests_by_seed=pairs.to_dict(orient="records"))
save_json(OUT / "evaluation_results.json", payload)

imgs = sorted(OUT.glob("*_profiles.png"))
panels = "".join(f'<details><summary>{html_lib.escape(p.stem)}</summary><img style="max-width:100%" '
                 f'src="data:image/png;base64,{base64.b64encode(p.read_bytes()).decode()}"/></details>' for p in imgs)
fmt = lambda df: df.to_html(index=False, escape=True, float_format=lambda x: f"{x:.4f}")
report = ("<!doctype html><html><head><meta charset='utf-8'><title>Praxis evaluation</title>"
          "<style>body{font:14px system-ui;margin:28px;color:#222}table{border-collapse:collapse;font-size:12px;margin:10px 0}"
          "th,td{padding:5px 8px;border:1px solid #ddd}h2{margin-top:28px}</style></head><body>"
          f"<h1>Praxis evaluation — Steps 7–9</h1><p>{html_lib.escape(EVAL_ID)}</p>"
          f"<p><b>Detection definition:</b> {html_lib.escape(DETECTION_DEFINITION)}; false-alarm target {FPR_TARGET:.0%} on validation benign records.</p>"
          "<h2>Summary</h2>" + fmt(summary) +
          "<h2>Detection counts — known and unknown attacks (selected runs)</h2>" + fmt(count_table) +
          "<h2>H1 — paired bootstrap</h2>" + fmt(h1_table.astype({c: str for c in ['dF1_known_CI', 'dF1_zd_CI', 'dG_CI']})) +
          "<h2>H2 — recall at common false-alarm rate</h2>" + fmt(h2_table) +
          "<h2>H1 — every seed (A and B paired by seed)</h2>" + fmt(h1_seeds.astype({c: str for c in ['dF1_known_CI', 'dF1_zd_CI', 'dG_CI']})) +
          "<h2>Held-out outcomes 1 and 2, every seed</h2>" + fmt(outcomes) +
          "<h2>Paired tests, every seed</h2>" + fmt(pairs) +
          "<h2>Synthetic-OOD output (descriptive, Experiment B)</h2>" + fmt(ood_table) +
          "<h2>Per-run metrics</h2>" + fmt(runs.drop(columns=["detection_definition"])) +
          "<h2>H3 — SHAP similarity with controls</h2>" + (fmt(h3) if len(h3) else "<p>Not run.</p>") + panels +
          "</body></html>")
(OUT / "evaluation_report.html").write_text(report, encoding="utf-8")
display(HTML(report))
print("✓ Evaluation complete:", OUT)